# Person 4 — Grouped model comparison and selection

Part of the six-person Basil Leaf ML pipeline. Run the numbered notebooks in order. This notebook states its inputs, produces a concrete handoff in `parts/artifacts`, and does not overwrite the complete project's `outputs/` results.

## Responsibility
Compare the baseline, Logistic Regression, SVM, Random Forest, and KNN with three grouped cross-validation folds. Choose by mean validation macro-F1 without looking at test performance.

**Input:** Person 2 manifest and Person 3 features  
**Output:** `04_model_comparison.csv`, `04_selection.json`, and `04_cv_folds.csv`

In [1]:
from pathlib import Path
import json, sys
import numpy as np
import pandas as pd

HERE = Path.cwd().resolve()
ROOT = next((p for p in (HERE, *HERE.parents) if (p / "Basil_Leaf_ML_Workflow.ipynb").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Run from the project folder or parts folder.")
DATA_DIR = ROOT / "data" / "raw"
ARTIFACTS = ROOT / "parts" / "artifacts"
ARTIFACTS.mkdir(parents=True, exist_ok=True)
SEED = 42
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tif", ".tiff"}
FOLDERS = {
    "Amravati_Region_Basil_Plant_Healthy": ("Healthy", 31),
    "Nagpur_Region_Basil_Plant_Healthy": ("Healthy", 473),
    "Pune_Region_Basil_Plant_Healthy": ("Healthy", 146),
    "Basil_Plant_Unhealthy": ("Unhealthy", 481),
}
print("Project:", ROOT)
print("Python:", sys.executable)
import time
from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

Project: D:\SLIIT\projectr\Dataset_Train
Python: D:\SLIIT\projectr\Dataset_Train\.venv\Scripts\python.exe


In [2]:
manifest=pd.read_csv(ARTIFACTS/'02_clean_manifest.csv')
X=np.load(ARTIFACTS/'03_features.npz')['X']; y=manifest.label.to_numpy()
dev=np.flatnonzero(manifest.split.eq('development')); groups=manifest.split_group.to_numpy()
cv=list(StratifiedGroupKFold(3,shuffle=True,random_state=SEED).split(dev,y[dev],groups[dev]))
models=[('Majority baseline',DummyClassifier(strategy='most_frequent'))]
for c in (.1,1.0): models.append((f'Logistic regression C={c}',make_pipeline(StandardScaler(),LogisticRegression(C=c,max_iter=3000,class_weight='balanced',random_state=SEED))))
for c in (1.0,10.0): models.append((f'RBF SVM C={c}',make_pipeline(StandardScaler(),SVC(C=c,class_weight='balanced',random_state=SEED))))
models += [('Random forest',RandomForestClassifier(n_estimators=200,min_samples_leaf=2,max_features='sqrt',class_weight='balanced',random_state=SEED,n_jobs=2)),('5-nearest neighbors',make_pipeline(StandardScaler(),KNeighborsClassifier(n_neighbors=5)))]
rows=[]; fold_rows=[]
for name,model in models:
    f1s=[]; accs=[]; times=[]
    for fold,(tr_rel,va_rel) in enumerate(cv,1):
        tr,va=dev[tr_rel],dev[va_rel]
        if set(groups[tr])&set(groups[va]): raise AssertionError('Group leakage')
        fitted=clone(model); start=time.perf_counter(); fitted.fit(X[tr],y[tr]); times.append(time.perf_counter()-start)
        pred=fitted.predict(X[va]); f1s.append(f1_score(y[va],pred,average='macro',zero_division=0)); accs.append(accuracy_score(y[va],pred))
        fold_rows.append({'model':name,'fold':fold,'train':len(tr),'validation':len(va),'macro_f1':f1s[-1]})
    rows.append({'model':name,'cv_macro_f1':np.mean(f1s),'cv_std':np.std(f1s),'cv_accuracy':np.mean(accs),'mean_fit_seconds':np.mean(times)})
comparison=pd.DataFrame(rows).sort_values(['cv_macro_f1','mean_fit_seconds','model'],ascending=[False,True,True]).reset_index(drop=True)
winner=comparison.iloc[0]
comparison.to_csv(ARTIFACTS/'04_model_comparison.csv',index=False); pd.DataFrame(fold_rows).to_csv(ARTIFACTS/'04_cv_folds.csv',index=False)
selection={'selected_model':winner.model,'selection_metric':'mean grouped 3-fold macro-F1','validation_macro_f1':float(winner.cv_macro_f1),'rule':'Highest mean macro-F1; test set excluded.'}
(ARTIFACTS/'04_selection.json').write_text(json.dumps(selection,indent=2),encoding='utf-8')
display(comparison); print(selection)

,model,cv_macro_f1,cv_std,cv_accuracy,mean_fit_seconds
0,Random forest,0.963648,5.248333e-03,0.963738,1.027501
1,RBF SVM C=10.0,0.953943,1.564970e-02,0.953975,0.246481
2,RBF SVM C=1.0,0.938557,1.687733e-02,0.938633,0.267131
3,Logistic regression C=1.0,0.917710,7.111182e-03,0.917713,0.044287
4,Logistic regression C=0.1,0.914918,5.218036e-03,0.914923,0.122495
5,5-nearest neighbors,0.891194,1.806359e-02,0.891213,0.019585
6,Majority baseline,0.337950,5.551115e-17,0.510460,0.002937


{'selected_model': 'Random forest', 'selection_metric': 'mean grouped 3-fold macro-F1', 'validation_macro_f1': 0.9636475644200724, 'rule': 'Highest mean macro-F1; test set excluded.'}


## Handoff to Person 5
Report all candidate results, not only the winner. Person 5 receives the selected model name and fits a fresh copy on the entire development set.